Phase 2: Feature Engineering

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('../data/processed/nsei_cleaned.csv', parse_dates=['Date'], index_col='Date')
print(f"Loaded cleaned data: {df.shape}")
display(df.head())

Loaded cleaned data: (1667, 8)


,Close,High,Low,Open,Volume,Return,High_Low_Range,Volume_Change
Date,,,,,,,,
2020-01-02,12282.200195,12289.900391,12195.250000,12198.549805,407700,0.008151,94.650391,0.340677
2020-01-03,12226.650391,12265.599609,12191.349609,12261.099609,428800,-0.004533,74.250000,0.051754
2020-01-06,11993.049805,12179.099609,11974.200195,12170.599609,396500,-0.019291,204.899414,-0.075326
2020-01-07,12052.950195,12152.150391,12005.349609,12079.099609,447800,0.004982,146.800781,0.129382
2020-01-08,12025.349609,12044.950195,11929.599609,11939.099609,446000,-0.002293,115.350586,-0.004020


2.1 Lag Features

In [3]:
# Lagged returns (past 1, 2, 3, 5, 10 days)
for lag in [1, 2, 3, 5, 10]:
    df[f'Return_lag{lag}'] = df['Return'].shift(lag)

# Lagged absolute returns (proxy for recent volatility)
for lag in [1, 2, 3, 5]:
    df[f'AbsReturn_lag{lag}'] = df['Return'].abs().shift(lag)

print("Lag features created.")
display(df[['Return', 'Return_lag1', 'Return_lag2', 'AbsReturn_lag1']].head(10))

Lag features created.


,Return,Return_lag1,Return_lag2,AbsReturn_lag1
Date,,,,
2020-01-02,0.008151,NaN,NaN,NaN
2020-01-03,-0.004533,0.008151,NaN,0.008151
2020-01-06,-0.019291,-0.004533,0.008151,0.004533
2020-01-07,0.004982,-0.019291,-0.004533,0.019291
2020-01-08,-0.002293,0.004982,-0.019291,0.004982
2020-01-09,0.015722,-0.002293,0.004982,0.002293
2020-01-10,0.003342,0.015722,-0.002293,0.015722
2020-01-13,0.005918,0.003342,0.015722,0.003342
2020-01-14,0.002653,0.005918,0.003342,0.005918


2.2 Rolling Volatility Features

In [4]:
for window in [5, 10, 20]:
    df[f'Vol_rolling_{window}'] = df['Return'].rolling(window=window).std()

# Also add rolling mean of absolute returns (alternative vol proxy)
df['AbsReturn_roll_5'] = df['Return'].abs().rolling(window=5).mean()

print("Rolling volatility features created.")
display(df[['Vol_rolling_5', 'Vol_rolling_10', 'Vol_rolling_20', 'AbsReturn_roll_5']].tail())

Rolling volatility features created.


,Vol_rolling_5,Vol_rolling_10,Vol_rolling_20,AbsReturn_roll_5
Date,,,,
2026-09-24,0.008859,0.007365,0.005729,0.006269
2026-09-25,0.008873,0.007451,0.005839,0.006288
2026-09-28,0.010218,0.008715,0.006431,0.008846
2026-09-29,0.010262,0.008082,0.006427,0.008679
2026-09-30,0.008631,0.007821,0.006417,0.008516


2.3 Technical Indicators

In [5]:
# 2.3a: Moving Averages
for window in [5, 20, 50]:
    df[f'MA_{window}'] = df['Close'].rolling(window=window).mean()
    
# Distance of price from MA (normalized) — often more informative than raw MA
df['Close_MA20_ratio'] = df['Close'] / df['MA_20']
df['Close_MA50_ratio'] = df['Close'] / df['MA_50']

print("Moving averages created.")
display(df[['Close', 'MA_5', 'MA_20', 'MA_50', 'Close_MA20_ratio']].tail())

Moving averages created.


,Close,MA_5,MA_20,MA_50,Close_MA20_ratio
Date,,,,,
2026-09-24,23063.099609,23319.920312,23600.817480,24013.100000,0.977216
2026-09-25,23140.500000,23278.740234,23553.300000,23994.455000,0.982474
2026-09-28,22780.250000,23151.930078,23483.529980,23963.373984,0.970052
2026-09-29,22716.199219,23029.369922,23415.319922,23932.927969,0.970143
2026-09-30,22620.449219,22864.099609,23343.552344,23901.582969,0.969023


In [6]:
# 2.3b: RSI 14
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)
df['RSI_14'] = 100 - (100 / (1 + rs))

print("RSI(14) created.")
display(df[['Close', 'RSI_14']].tail())

RSI(14) created.


,Close,RSI_14
Date,,
2026-09-24,23063.099609,27.217597
2026-09-25,23140.500000,29.329631
2026-09-28,22780.250000,25.910452
2026-09-29,22716.199219,26.950350
2026-09-30,22620.449219,28.491931


In [7]:
# 2.3c: MACD
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()

df['MACD'] = ema_12 - ema_26
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_hist'] = df['MACD'] - df['MACD_signal']

print("MACD created.")
display(df[['Close', 'MACD', 'MACD_signal', 'MACD_hist']].tail())

MACD created.


,Close,MACD,MACD_signal,MACD_hist
Date,,,,
2026-09-24,23063.099609,-240.878358,-222.193131,-18.685227
2026-09-25,23140.500000,-244.145403,-226.583586,-17.561817
2026-09-28,22780.250000,-272.660661,-235.799001,-36.861660
2026-09-29,22716.199219,-297.003895,-248.039979,-48.963915
2026-09-30,22620.449219,-320.329746,-262.497933,-57.831813


In [8]:
# 2.3d: ATR 14
high_low = df['High'] - df['Low']
high_close = (df['High'] - df['Close'].shift(1)).abs()
low_close = (df['Low'] - df['Close'].shift(1)).abs()

true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['ATR_14'] = true_range.rolling(window=14).mean()

print("ATR(14) created.")
display(df[['High', 'Low', 'Close', 'ATR_14']].tail())

ATR(14) created.


,High,Low,Close,ATR_14
Date,,,,
2026-09-24,23281.949219,23046.150391,23063.099609,202.725167
2026-09-25,23162.699219,23020.949219,23140.500000,203.400112
2026-09-28,23080.250000,22762.199219,22780.250000,219.007394
2026-09-29,22753.250000,22569.650391,22716.199219,222.903739
2026-09-30,22809.349609,22595.199219,22620.449219,223.657366


2.4 Volume-Based Features

In [9]:
df['Volume_MA_5'] = df['Volume'].rolling(window=5).mean()
df['Volume_MA_20'] = df['Volume'].rolling(window=20).mean()
df['Volume_ratio'] = df['Volume'] / df['Volume_MA_20']

print("Volume-based features created.")
display(df[['Volume', 'Volume_MA_5', 'Volume_MA_20', 'Volume_ratio']].tail())

Volume-based features created.


,Volume,Volume_MA_5,Volume_MA_20,Volume_ratio
Date,,,,
2026-09-24,271700,268460.0,299575.0,0.906952
2026-09-25,242700,241940.0,295540.0,0.821209
2026-09-28,261500,251520.0,293810.0,0.890031
2026-09-29,603200,323340.0,288425.0,2.091358
2026-09-30,379000,351620.0,290665.0,1.303907


2.5 Define Target Variable
Target = next-day 5-day forward volatility.

We compute the std of returns over the next 5 days (t+1 to t+5). We use shift(-5) on the rolling std, so at time *t* the value is the std of returns at [t+1, t+5]. This is what the model will try to predict at time *t*.

In [10]:
# Rolling std of returns, then shift backwards so it aligns with the *current* row
forward_window = 5
df['Target_Volatility'] = df['Return'].rolling(window=forward_window).std().shift(-forward_window)

print(f"Target = {forward_window}-day forward volatility")
display(df[['Return', 'Target_Volatility']].tail(10))

Target = 5-day forward volatility


,Return,Target_Volatility
Date,,
2026-09-17,0.002280,0.008859
2026-09-18,0.003252,0.008873
2026-09-21,0.002904,0.010218
2026-09-22,-0.003650,0.010262
2026-09-23,0.005037,0.008631
2026-09-24,-0.016500,NaN
2026-09-25,0.003350,NaN
2026-09-28,-0.015690,NaN
2026-09-29,-0.002816,NaN


In [11]:
# Code - Drop NaNs & Check
print(f"Shape before dropping NaNs: {df.shape}")
df = df.dropna()
print(f"Shape after dropping NaNs: {df.shape}")
print(f"Date range: {df.index.min()} → {df.index.max()}")

Shape before dropping NaNs: (1667, 35)
Shape after dropping NaNs: (1612, 35)
Date range: 2020-03-13 00:00:00 → 2026-09-23 00:00:00


Feature Set Summary
Here is the final set of engineered features. The target is Target_Volatility.

In [12]:
feature_cols = [c for c in df.columns if c not in ['Target_Volatility']]
print(f"Total features: {len(feature_cols)}")
print("\nFeature list:")
for f in feature_cols:
    print(f" - {f}")

Total features: 34

Feature list:
 - Close
 - High
 - Low
 - Open
 - Volume
 - Return
 - High_Low_Range
 - Volume_Change
 - Return_lag1
 - Return_lag2
 - Return_lag3
 - Return_lag5
 - Return_lag10
 - AbsReturn_lag1
 - AbsReturn_lag2
 - AbsReturn_lag3
 - AbsReturn_lag5
 - Vol_rolling_5
 - Vol_rolling_10
 - Vol_rolling_20
 - AbsReturn_roll_5
 - MA_5
 - MA_20
 - MA_50
 - Close_MA20_ratio
 - Close_MA50_ratio
 - RSI_14
 - MACD
 - MACD_signal
 - MACD_hist
 - ATR_14
 - Volume_MA_5
 - Volume_MA_20
 - Volume_ratio


In [13]:
output_path = '../data/processed/nsei_features.csv'
df.to_csv(output_path)
print(f"Phase 2 complete! Features saved to {output_path}")

Phase 2 complete! Features saved to ../data/processed/nsei_features.csv
